# 02 · Publish results safely

The default is a non-mutating Git dry-run. It discovers the latest compatible
completed final run, creates or reuses a lightweight bundle, validates it,
scans for secrets and forbidden artifacts, and previews the exact Git paths.
Publishing requires both explicit switches below.


In [ ]:
MODEL_ID = "rtdetrv2_l"

PUBLISH_RESULTS = False
DRY_RUN = True


In [ ]:
import importlib.util
import json
import os
import subprocess
import sys
from pathlib import Path

try:
    IN_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    IN_COLAB = False
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
REPO_PATH = Path("/content/aerial-object-detection-benchmark") if IN_COLAB else Path.cwd()
if IN_COLAB:
    if not (REPO_PATH / ".git").is_dir():
        subprocess.run(["git", "clone", "--branch", "main", "https://github.com/Harryphan72007/aerial-object-detection-benchmark.git", str(REPO_PATH)], check=True)
    elif subprocess.check_output(["git", "-C", str(REPO_PATH), "status", "--porcelain"], text=True).strip():
        raise RuntimeError("Repository has local changes; refusing to update it.")
    else:
        subprocess.run(["git", "-C", str(REPO_PATH), "pull", "--ff-only", "origin", "main"], check=True)
sys.path.insert(0, str(REPO_PATH))
DRIVE_ROOT = (
    Path("/content/drive/MyDrive/visdrone_architecture_benchmark")
    if IN_COLAB
    else Path(os.environ.get("VISDRONE_DRIVE_ROOT", REPO_PATH / "local_artifacts"))
)
SMOKE_TEST = os.environ.get("SMOKE_TEST", "").lower() in {"1", "true", "yes"}


In [ ]:
if SMOKE_TEST:
    result = {
        "dry_run": DRY_RUN,
        "publish_results": PUBLISH_RESULTS,
        "message": "Smoke mode validates notebook control flow without requiring completed GPU artifacts.",
    }
else:
    from src.workflows.publishing import publish_results
    result = publish_results(
        REPO_PATH,
        DRIVE_ROOT,
        MODEL_ID,
        publish_results=PUBLISH_RESULTS,
        dry_run=DRY_RUN,
    )
print(json.dumps(result, indent=2, default=str))


In [ ]:
if DRY_RUN:
    print("DRY RUN COMPLETE — Git and GitHub were not modified.")
    print("Review included/excluded files, size, target paths, and Git preview above.")
else:
    print(f"Published bundle: {result['bundle_id']}")
    print(f"Pull request: {result.get('pull_request') or 'Open/report manually from experiment-results'}")
